Topography Relaxation with surface processes
======

This notebook models the topography relaxation with surface processes (Model TR-CM2) as described in Neng et al. (2026). It integrates Underworld2 with Badlands using the UWGeodynamic module within the ALE-IB scheme.

**References**

Lu, N., Moresi, L., Giordani, J., & Knight, B. (2026). A novel ALE scheme with the internal boundary for coupling tectonic and surface processes in geodynamic models. EGUsphere, 2026, 1-35.

In [ ]:
import underworld as uw
import math
from underworld import function as fn
import numpy as np
import os

from underworld import UWGeodynamics as GEO
u = GEO.UnitRegistry
ndim = GEO.non_dimensionalise
dimen = GEO.dimensionalise

comm = uw.mpi.comm
rank = uw.mpi.rank
size = uw.mpi.size

GEO.rcParams["initial.nonlinear.tolerance"] = 1e-3
GEO.rcParams['initial.nonlinear.max.iterations'] = 100
GEO.rcParams["nonlinear.tolerance"] = 1e-3
GEO.rcParams['nonlinear.max.iterations'] = 100
GEO.rcParams["popcontrol.particles.per.cell.2D"] = 20
GEO.rcParams["swarm.particles.per.cell.2D"] = 20
GEO.rcParams["surface.pressure.normalization"] = True
GEO.rcParams["pressure.smoothing"] = True
GEO.rcParams["popcontrol.split.threshold"] = 0.1

half_rate = 1.0 * u.centimeter / u.year
model_length = 500. * u.kilometer
gravity = 9.81 * u.meter / u.second**2
bodyforce = 3300 * u.kilogram / u.metre**3 *gravity 

KL = model_length
Kt = KL / half_rate
KM = bodyforce * KL**2 * Kt**2

GEO.scaling_coefficients["[length]"] = KL
GEO.scaling_coefficients["[time]"] = Kt
GEO.scaling_coefficients["[mass]"]= KM

In [ ]:
longtest = False
dy = ndim(10.0 * u.kilometer)
max_time = 11.0*u.kiloyear

if "UW_LONGTEST" in os.environ or longtest:
    dy = ndim(2.5 * u.kilometer)
    max_time = 301.0*u.kiloyear

xmin, xmax = ndim(-160 * u.kilometer), ndim(160 * u.kilometer)
ymin, ymax = ndim(-160 * u.kilometer), ndim(40 * u.kilometer)
yint = 0.
 
dy = ndim(2.5 * u.kilometer)
dx = dy
xRes,yRes = int(np.around((xmax-xmin)/dx)),int(np.around((ymax-ymin)/dy))
yResa,yResb =int(np.around((ymax-yint)/dy)),int(np.around((yint-ymin)/dy))

Model = GEO.Model(elementRes=(xRes, yRes),
                  minCoord=(xmin,ymin),
                  maxCoord=(xmax, ymax),
                  gravity=(0.0, -gravity),
                  periodic=(True, False))
Model.outputDir= "1_23_08_CouplingModellingALEIB_TopographyRelaxation_yres{:n}".format(yRes)
Model.minStrainRate = 1e-18 / u.second

wRatio = 1
D = np.abs(ymin)
Lambda = D/wRatio
k = 2.0 * np.pi / Lambda
mu0 = ndim(1e21  * u.pascal * u.second)
g = ndim(gravity)
rho0 = ndim(3300* u.kilogram / u.metre**3)
drho = rho0-0.
w_m = ndim(5*u.kilometer)

tau0 = 2*k*mu0/drho/g
tau = (D*k+np.sinh(D*k)*np.cosh(D*k))/(np.sinh(D*k)**2)*tau0

fn_coord = fn.input()
surf_fn = w_m * fn.math.cos(2.*np.pi*fn_coord[0]/Lambda) 

Model.inter_wall = Model._get_InternalwallSets(yint)
with Model.mesh.deform_mesh():
     Model.mesh.data[Model.inter_wall.data, 1] = surf_fn.evaluate(Model.inter_wall)[:,0]
Model._freeSurface_ALEIB = True 
Model.freeSurface = True 
Model._freeSurface.solve(0.)

# recreate swarm as mesh deformed
import underworld as uw
from collections import OrderedDict 
Model.swarm_variables = OrderedDict()
Model.swarm = uw.swarm.Swarm(mesh=Model.mesh, particleEscape=True)
Model.swarm.allow_parallel_nn = True 
particlesPerCell = GEO.rcParams["swarm.particles.per.cell.2D"]
Model._swarmLayout = uw.swarm.layouts.PerCellSpaceFillerLayout(swarm=Model.swarm,particlesPerCell=particlesPerCell)
Model.swarm.populate_using_layout(layout=Model._swarmLayout)
Model._initialize()

materialAShape = fn_coord[1] > surf_fn
materialMShape = fn_coord[1] <= surf_fn

materialA = Model.add_material(name="Air", shape=materialAShape)
materialM = Model.add_material(name="Mantle", shape=materialMShape) 

materialA.viscosity = 1e18 * u.pascal * u.second
materialM.viscosity = 1e21 * u.pascal * u.second

materialA.density = 0.
materialM.density = 3300 * u.kilogram / u.metre**3

sediment = Model.add_material(name="sediment")
sediment.viscosity = 1e19 * u.pascal * u.second
sediment.density = 2700 * u.kilogram / u.metre**3 

Model.set_velocityBCs(left=[0.,None],right=[0,None],bottom=[0.,0.], top=[None, 0.])
Model.init_model()

Model.solver.set_inner_method("mumps")
Model.solver.set_penalty(1e3)

In [ ]:
dt_set = 2.5*u.kiloyear
checkpoint_interval = 10.0*u.kiloyear
Model.surfaceProcesses = GEO.surfaceProcesses.Badlands(airIndex=[materialA.index],sedimentIndex=sediment.index,XML="resources/badlands_1en5.xml", resolution=1.25 * u.kilometre, checkpoint_interval=dt_set,aspectRatio2d=0.25,surfElevation=surf_fn)

In [ ]:
Model.run_for(max_time, checkpoint_interval=checkpoint_interval,dt=dt_set)